**CI twin of `ch09-micrograd-backward.qmd`.** Generated by `infra/ci/make_twin.py` (P1-D10); do not edit by hand. It runs the chapter's worked example and exercise solutions under CPython so the R10 gate proves they work (DECISIONS D0010).

In [ ]:
# Make the single-sourced grader importable under CPython (no paste; P1-D9).
import sys, pathlib
for _p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
    if (_p / 'lib' / 'grader.py').exists():
        sys.path.insert(0, str(_p))
        break
from lib.grader import run_tests

In [ ]:
class V8:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _prev
        self._op = _op

    def __add__(self, other):
        out = V8(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = V8(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def relu(self):
        out = V8(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        return out

def neuron():
    x1, w1, x2, w2 = V8(1.0), V8(0.5), V8(0.5), V8(-0.5)
    p1 = x1 * w1; p2 = x2 * w2
    z = p1 + p2; out = z.relu()
    return x1, w1, x2, w2, p1, p2, z, out

print("engine reassembled")

In [ ]:
x1, w1, x2, w2, p1, p2, z, out = neuron()
out.grad = 1.0
out._backward(); p1._backward(); z._backward(); p2._backward()
print(f"wrong order:  w1.grad = {w1.grad}   x1.grad = {x1.grad}")

x1, w1, x2, w2, p1, p2, z, out = neuron()
out.grad = 1.0
out._backward(); z._backward(); p1._backward(); p2._backward()
print(f"right order:  w1.grad = {w1.grad}   x1.grad = {x1.grad}")

In [ ]:
def topo(root):
    order, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for parent in v._prev:
                build(parent)
            order.append(v)
    build(root)
    return order

x1, w1, x2, w2, p1, p2, z, out = neuron()
order = topo(out)
print([v._op if v._op else "leaf" for v in order])

In [ ]:
out.grad = 1.0
for v in reversed(order):
    v._backward()

print(f"w1.grad = {w1.grad}   w2.grad = {w2.grad}")
print(f"x1.grad = {x1.grad}   x2.grad = {x2.grad}")

In [ ]:
import math

class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _prev
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def __pow__(self, k):
        out = Value(self.data ** k, (self,), f"**{k}")
        def _backward():
            self.grad += out.grad * k * self.data ** (k - 1)
        out._backward = _backward
        return out

    def exp(self):
        out = Value(math.exp(self.data), (self,), "exp")
        def _backward():
            self.grad += out.grad * out.data
        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        return out

    # ergonomics: compositions and scalar-friendliness, no new calculus
    def __neg__(self):           return self * -1
    def __sub__(self, other):    return self + (-other if isinstance(other, Value) else -other)
    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1
    def __radd__(self, other):   return self + other
    def __rmul__(self, other):   return self * other
    def __rsub__(self, other):   return (-self) + other

    def backward(self):
        order, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for parent in v._prev:
                    build(parent)
                order.append(v)
        build(self)
        self.grad = 1.0             # blame starts at the output, in full
        for v in reversed(order):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad})"

print("micrograd complete")

In [ ]:
def sigmoid(z):
    return (1 + (-z).exp()) ** -1

xa, xb = Value(1.0), Value(0.5)
w11, w12, b1 = Value(0.5), Value(-0.5), Value(0.0)
w21, w22, b2 = Value(1.0), Value(1.0), Value(-1.0)
w31, w32, b3 = Value(1.0), Value(-1.0), Value(0.5)

h1 = sigmoid(xa * w11 + xb * w12 + b1)
h2 = sigmoid(xa * w21 + xb * w22 + b2)
out = sigmoid(h1 * w31 + h2 * w32 + b3)
loss = (out - 1.0) ** 2

loss.backward()                       # the entire ceremony

print(f"loss = {loss.data:.4f}   (Ch6: 0.1535)")
print(f"w1: {w11.grad:.5f}, {w12.grad:.5f}   b1: {b1.grad:.5f}")
print(f"w2: {w21.grad:.5f}, {w22.grad:.5f}   b2: {b2.grad:.5f}")
print(f"w3: {w31.grad:.4f}, {w32.grad:.4f}   b3: {b3.grad:.4f}")
print(f"x1: {xa.grad:.5f}  ← the two-path sum, automatic")

In [ ]:
from lib.grader import run_tests, grad_check

def sigf(z):
    return 1 / (1 + math.exp(-z))

def loss_of(params):
    a, b, c, d, e, f, g, h, i = params
    hh1 = sigf(a * 1.0 + b * 0.5 + c)
    hh2 = sigf(d * 1.0 + e * 0.5 + f)
    return (sigf(g * hh1 + h * hh2 + i) - 1.0) ** 2

run_tests([
    grad_check("finished engine, all nine parameters", loss_of,
               [0.5, -0.5, 0.0, 1.0, 1.0, -1.0, 1.0, -1.0, 0.5],
               [w11.grad, w12.grad, b1.grad, w21.grad, w22.grad, b2.grad,
                w31.grad, w32.grad, b3.grad]),
])

print("nodes in this graph:", len(topo(loss)))

In [ ]:
print(f"b3.grad after one backward :  {b3.grad:.4f}   (the truth)")

loss.backward()                       # second pass, grads not zeroed
print(f"b3.grad after two backwards:  {b3.grad:.4f}   (~8× the truth!)")

def zero_grad(root):
    order, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for parent in v._prev:
                build(parent)
            order.append(v)
    build(root)
    for v in order:
        v.grad = 0.0

zero_grad(loss)
loss.backward()
print(f"zeroed, then one backward  :  {b3.grad:.4f}   (sanity restored)")

In [ ]:
class N:
    def __init__(self, data, _prev=()):
        self.data = data
        self._prev = _prev

def topo(root):
    order, visited = [], set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for parent in v._prev:
                build(parent)
            order.append(v)
    build(root)
    return order

x1, w1, x2, w2 = N(1.0), N(0.5), N(0.5), N(-0.5)
p1 = N(0.5, (x1, w1)); p2 = N(-0.25, (x2, w2))
z = N(0.25, (p1, p2)); out = N(0.25, (z,))

order = topo(out)
parents_first = all(order.index(p) < order.index(v)
                    for v in order for p in v._prev)

run_tests([
    ("every node recorded exactly once", len(order), 8),
    ("every node appears after its parents", parents_first, True),
])

In [ ]:
class V:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = _prev
        self._op = _op

    def __add__(self, other):
        out = V(self.data + other.data, (self, other), "+")
        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward
        return out

    def __mul__(self, other):
        out = V(self.data * other.data, (self, other), "*")
        def _backward():
            self.grad += out.grad * other.data
            other.grad += out.grad * self.data
        out._backward = _backward
        return out

    def relu(self):
        out = V(self.data if self.data > 0 else 0.0, (self,), "relu")
        def _backward():
            self.grad += out.grad * (1.0 if self.data > 0 else 0.0)
        out._backward = _backward
        return out

    def backward(self):
        order, visited = [], set()
        def build(v):
            if v not in visited:
                visited.add(v)
                for parent in v._prev:
                    build(parent)
                order.append(v)
        build(self)
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

x1, w1, x2, w2 = V(1.0), V(0.5), V(0.5), V(-0.5)
out = (x1 * w1 + x2 * w2).relu()
out.backward()

a = V(3.0)
c = a * a
c.backward()

def neuron_of(params):
    wa, wb = params
    return max(0.0, 1.0 * wa + 0.5 * wb)

run_tests([
    ("neuron weight blames", [w1.grad, w2.grad], [1.0, 0.5]),
    ("neuron input blames", [x1.grad, x2.grad], [0.5, -0.5]),
    ("shared node: both paths delivered", a.grad, 6.0),
    grad_check("your backward() vs finite differences",
               neuron_of, [0.5, -0.5], [w1.grad, w2.grad]),
])